# Preparing an oscillator ground state on the position grid

Start from the two grid points nearest zero, turn on the kinetic term and then ramp the anharmonic interaction. Compare the final state with the ground state of the same finite Hamiltonian. This accompanies Chapters 9 and 10 and follows [the grid-encoding example](t6.ipynb). The final NumPy comparison has a repaired variable name and must be rerun.

[Qiskit tutorial](../../../../python-docs/qiskit.md)


In [15]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.circuit import Parameter, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFTGate
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

# Ground state preparation

To prepare the ground state we will start from the $x^2$ Hamiltonian.
$H_{\text{initial}} = x^2 / 2$
We want the final Hamiltonian
\begin{equation}
H_1 = H_{\text{initial}} + \frac{p^2}{2}
\end{equation}

We will dial up a piece in front of $p^2$.
\begin{equation}
H_{u} = \frac{x^2}{2} + r(u) \frac{p^2}{2}
\end{equation}

On the finite position grid, the minimum of $x^2/2$ occurs at the two samples nearest zero. We choose the even combination in this degenerate initial subspace.


Because $N$ is even, zero lies halfway between those samples. Their coordinates are $x=\pm\delta_x/2$.


The two indices are $j=N/2-1$ and $j=N/2$. Substitution into $x_j=\delta_x[j-(N-1)/2]$ gives $-\delta_x/2$ and $+\delta_x/2$, respectively.


The even initial state is

$$
\ket{\psi_{\rm initial}}=
\frac{\ket{j=N/2-1}+\ket{j=N/2}}{\sqrt2}.
$$

Their bitstrings are $0\,1\cdots1$ and $1\,0\cdots0$. The following circuit creates their equal superposition by using the most significant qubit as a control. The interpolation preserves parity, so this choice follows the even sector as the kinetic term is turned on.


In [2]:
def prep_midpoint_even(n):
    """
    Prepare (|0 1 1 ... 1> + |1 0 0 ... 0>)/sqrt(2) on n qubits.
    We take qubit indexing as: q[0]=LSB, q[n-1]=MSB.
    """
    qc = QuantumCircuit(n)
    ctrl = n-1  # MSB

    # Put MSB in (|0> + |1>)/sqrt(2)
    qc.h(ctrl)

    # For each lower qubit, flip iff MSB == 0 (open-controlled X)
    for tgt in range(n-1):  # 0 .. n-2
        qc.x(ctrl)
        qc.cx(ctrl, tgt)
        qc.x(ctrl)

    return qc

def prep_midpoint_odd(n):
    qc = prep_midpoint_even(n)
    # Add relative phase: (|...> - |...>)/sqrt(2)
    qc.z(n-1)  # MSB
    return qc

In [3]:
def strang_step_gs_prep(qc: QuantumCircuit, lam: float, delta_t: float,
                x_max: float) -> None:
    # first x^2
    x2_term(qc, 0, delta_t / 2, x_max)
    # then apply the p^2 piece
    p2_term(qc, lam, delta_t, x_max)
    # then apply x^2 again
    x2_term(qc, 0, delta_t / 2, x_max)

def build_circuit_gs_prep(num_qubits, x_max, tPrep=60.0, nSteps=600):
    qc = prep_midpoint_even(num_qubits)
    dt = tPrep / nSteps
    for j in range(1, nSteps + 1):
        u = (j - 0.5) / nSteps
        cpl  = r(u)
        strang_step_gs_prep(qc, cpl, dt, x_max)
    return qc

# Jordan-Lee-Preskill encoding

Let $n_q$ be the number of qubits and $N=2^{n_q}$ the number of position samples. Qubit $r=0$ is the least significant bit. The position grid is

$$
x_j=-x_{\max}+j\delta_x,\qquad
\delta_x=\frac{2x_{\max}}{N-1},\qquad j=0,\ldots,N-1.
$$

Write $j=\sum_{r=0}^{n_q-1}2^r b_r$, where $b_r$ is zero or one. Since $Z_r\ket{j}=(1-2b_r)\ket{j}$,

$$
\sum_r2^rZ_r\ket{j}=[(N-1)-2j]\ket{j}.
$$

Multiplication by $-x_{\max}/(N-1)$ gives $x_j$, so

$$
\hat x=-\frac{x_{\max}}{N-1}\sum_{r=0}^{n_q-1}2^r Z_r.
$$

We use the half-shifted Fourier grid

$$
p_k=-\frac{\pi}{\delta_x}+\left(k+\frac12\right)\frac{2\pi}{N\delta_x}
=\frac{\pi}{N\delta_x}[2k-(N-1)].
$$

The same binary identity gives the diagonal operator in the Fourier register,

$$
P_{\rm diag}=-\frac{\pi}{N\delta_x}\sum_{r=0}^{n_q-1}2^rZ_r.
$$

The kinetic-energy operator on the position register is obtained by changing to this Fourier basis, applying $P_{\rm diag}^2/2$ and changing back. The next derivation specifies the phases of that basis change.


The Qiskit Fourier gate has matrix elements

$$
F_{kj}=\frac{1}{\sqrt N}\exp\left(\frac{2\pi i jk}{N}\right).
$$

For the shifted grids above we use $S_{kj}=N^{-1/2}\exp(i p_kx_j)$. Here $p_kx_j$ is already the dimensionless phase in units with $\hbar=1$. Substituting both grids gives

$$
\begin{aligned}
p_kx_j
&=\frac{2\pi}{N}\left(j-\frac{N-1}{2}\right)
                    \left(k-\frac{N-1}{2}\right)\\
&=\frac{2\pi jk}{N}-\frac{\pi(N-1)}{N}(j+k)
  +\frac{\pi(N-1)^2}{2N}.
\end{aligned}
$$

Define $\alpha=\pi(1/N-1)$ and $D_{jj}=e^{i\alpha j}$. Then

$$
S=e^{i\pi(N-1)^2/(2N)}DFD.
$$

The constant phase cancels in $S^\dagger P_{\rm diag}^2S$. To implement $D$, apply $R_z(\alpha 2^r)$ to each qubit. Indeed,

$$
\prod_{r=0}^{n_q-1}R_z(\alpha2^r)\ket{j}
=e^{-i\alpha(N-1)/2}e^{i\alpha j}\ket{j}.
$$

The first factor is another state-independent phase. Thus the code applies these rotations, `QFTGate`, the same rotations, the diagonal kinetic evolution, and the inverse operations in reverse order. The positive sign in $S$ matches Qiskit's Fourier gate. Reversing that Fourier convention reverses the momentum labels and leaves the kinetic energy unchanged.


In [4]:
def x2_term(qc: QuantumCircuit, m_ctr: float, delta_t: float, x_max: float) -> None:
    qubits = qc.num_qubits
    x_prefactor = - x_max / (2 ** qubits - 1)
    delta_x = 2 * x_max / (2 ** qubits - 1)
    # apply the x^2 evolution
    for i in range(qubits):
        for j in range(i + 1, qubits):
            qc.rzz(2 * 2 ** (i + j) * (1 + m_ctr) * x_prefactor ** 2 * delta_t, i, j)

In [5]:
def p2_term(qc: QuantumCircuit, lam0f: float, delta_t: float, x_max: float):
    qubits = qc.num_qubits
    delta_x = 2 * x_max / (2 ** qubits - 1)
    p_prefactor = - np.pi / (2 ** qubits * delta_x)
    # do a fourier transform
    alpha = (1 / 2 ** qubits - 1) * np.pi
    for i in range(qubits):
        qc.rz(alpha * 2 ** i, i)
    qc.append(QFTGate(qubits), range(qubits))
    for i in range(qubits):
        qc.rz(alpha * 2 ** i, i)
    # then apply the p^2 evolution now
    for i in range(qubits):
        for j in range(i + 1, qubits):
            qc.rzz(2 * 2 ** (i + j) * lam0f * p_prefactor ** 2 * delta_t, i, j)
    # apply the inverse fourier transform
    for i in range(qubits):
        qc.rz( - alpha * 2 ** i, i)
    qc.append(QFTGate(qubits).inverse(), range(qubits))
    for i in range(qubits):
        qc.rz( - alpha * 2 ** i, i)

For the anharmonic oscillator,

$$
H=\frac{p^2}{2}+\frac{x^2}{2}+\lambda x^4.
$$

Let $a=x_{\max}/(N-1)$ and $w_i=2^i$. Since $x=-a\sum_i w_iZ_i$, expanding its fourth power means sorting four indices by their repetitions. Four distinct indices occur in $4!=24$ orders. A repeated pair and two distinct remaining indices occur in $4!/2!=12$ orders. Three equal indices and one different index occur in four orders. Four equal indices, or two equal pairs, give the identity because $Z_i^2=I$. Consequently

$$
\begin{aligned}
x^4=a^4\Bigg[&
\left(\sum_i w_i^4+6\sum_{i<j}w_i^2w_j^2\right)I\\
&+24\sum_{i<j<k<l}w_iw_jw_kw_lZ_iZ_jZ_kZ_l\\
&+12\sum_i\sum_{\substack{j<k\\j,k\ne i}}w_i^2w_jw_kZ_jZ_k
+4\sum_i\sum_{j\ne i}w_i^3w_jZ_iZ_j\Bigg].
\end{aligned}
$$

All indices run from zero to $n_q-1$. This accounts for every term, including the identity contribution. The circuit below omits that contribution during ordinary state evolution because it changes only the global phase. For a controlled evolution in phase estimation or a Hadamard test, the phase must be restored, or the associated energy offset must be included explicitly. Each retained coefficient $c$ produces a rotation of angle $2c\,\delta t$, since $R_{ZZ}(\theta)=\exp(-i\theta Z_iZ_j/2)$. The CNOT ladder in `rz4` implements the same rule for four Pauli $Z$ operators.


In [6]:
def rz4(qc: QuantumCircuit, theta: float, i: int, j: int, k: int, l: int):
    qc.cx(i, j)
    qc.cx(j, k)
    qc.cx(k, l)
    qc.rz(theta, l)
    qc.cx(k, l)
    qc.cx(j, k)
    qc.cx(i, j)

In [7]:
def x4_term(qc: QuantumCircuit, lam: float, delta_t: float, x_max: float):
    qubits = qc.num_qubits
    x_prefactor = - x_max / (2 ** qubits - 1)
    delta_x = 2 * x_max / (2 ** qubits - 1)
    '''
    Apply the potential term.
    First we will apply the four-site Z interaction.
    '''
    for i in range(qubits):
        for j in range(i+1, qubits):
            for k in range(j+1, qubits):
                for l in range(k+1, qubits):
                    coeff = 2 ** (i+j+k+l) * 24 * lam * x_prefactor ** 4
                    rz4(qc, 2 * coeff * delta_t, i, j, k, l)
    '''
    Now we will apply the two-site term where two indices are equal.
    '''
    for i in range(qubits):
        for j in range(qubits):
            if j == i:
                continue
            for k in range(j+1, qubits):
                if k == i:
                    continue
                coeff = 2 ** (2 * i + j + k) * 12 * lam * x_prefactor ** 4
                qc.rzz(2 * coeff * delta_t, j, k)
    '''
    Finally apply the two site term where three indices are equal.
    '''
    for i in range(qubits):
        for j in range(qubits):
            if j == i:
                continue
            coeff = 2 ** (3 * i + j) * 4 * lam * x_prefactor ** 4
            qc.rzz(2 * coeff * delta_t, i, j)

In [8]:
def strang_step(qc: QuantumCircuit, lam: float, m_ctr: float, delta_t: float,
                x_max: float) -> None:
    # first apply x^4 and x^2
    x2_term(qc, m_ctr, delta_t / 2, x_max)
    x4_term(qc, lam, delta_t / 2, x_max)
    # then apply the p^2 piece
    p2_term(qc, 1, delta_t, x_max)
    # then apply x^4 and x^2 again
    x2_term(qc, m_ctr, delta_t / 2, x_max)
    x4_term(qc, lam, delta_t / 2, x_max)

In [9]:
# ramp terms
def r(u):
    return u * u * (3.0 - 2.0 * u)

def lamOf(u, lam):
    if 0.0 <= u <= 1.0:
        return lam * r(u)
    elif 1.0 < u <= 2.0:
        return lam
    return 0.0

def dm2Of(u, lam):
    if 0.0 <= u <= 1.0:
        return -6.0 * lam * r(u)
    elif 1.0 < u <= 2.0:
        return -6.0 * lam * (1.0 - r(u - 1.0))
    return 0.0

In [10]:
def build_circuit(lam, num_qubits, x_max, tPrep=25.0, nSteps=250):
    qc = build_circuit_gs_prep(num_qubits, x_max, 60.0, 600)
    dt = tPrep / (2.0 * nSteps)
    for j in range(1, 2 * nSteps + 1):
        u = (j - 0.5) / nSteps
        lam_t  = lamOf(u, lam)
        dm_ctr = dm2Of(u, lam)
        strang_step(qc, lam_t, dm_ctr, dt, x_max)
    return qc

# Running the circuit

In [44]:
lam = 0.5
num_qubits = 5
x_max = 10

In [45]:
qc = build_circuit(lam, num_qubits, x_max)

In [46]:
state = Statevector.from_instruction(qc)

# Compare with numpy

In [ ]:
# -------------------------
# Grids (your JLP formulas)
# -------------------------
def jlp_grids(n_qubits: int, x_max: float):
    N  = 2**n_qubits
    dx = 2.0 * x_max / (N - 1)
    j  = np.arange(N, dtype=float)
    # position grid (exactly N samples)
    x_vals = -x_max + j * dx
    # half-shifted momentum grid
    k_vals = -np.pi/dx + (j + 0.5) * (2.0*np.pi)/(N * dx)
    return x_vals, k_vals, dx

# ---------------------------------------
# Plane-wave "DFT" from your (k,x) grids
# ---------------------------------------
def plane_wave_S(n_qubits: int, x_max: float):
    x_vals, k_vals, _ = jlp_grids(n_qubits, x_max)
    # S_{l j} = (1/sqrt(N)) * exp(i k_l x_j)
    N = 2**num_qubitsubits
    S = np.exp(1j * np.outer(k_vals, x_vals)) / np.sqrt(N)
    return S

# -------------------------
# Operators and Hamiltonian
# -------------------------
def jlp_X_P(n_qubits: int, x_max: float):
    x_vals, k_vals, _ = jlp_grids(n_qubits, x_max)
    X = np.diag(x_vals.astype(complex))
    S = plane_wave_S(n_qubits, x_max)
    P = S.conj().T @ np.diag(k_vals.astype(complex)) @ S
    return X, P

def jlp_H(n_qubits: int, x_max: float, lam: float, m: float):
    X, P = jlp_X_P(n_qubits, x_max)
    X2 = X @ X
    return 0.5*(P@P) + 0.5*(1.0+m)*X2 + lam*(X2@X2)

def exact_diagonalize(n_qubits: int, x_max: float, lam: float, m: float):
    H = jlp_H(n_qubits, x_max, lam, m)
    evals, evecs = la.eigh(H)   # columns are eigenvectors
    return evals, evecs, H

# -------------------------
# Example / quick check
# -------------------------
if __name__ == "__main__":
    m     = 0.0 # mass counter term

    evals, evecs, H = exact_diagonalize(num_qubits, x_max, lam, m)
    print("Lowest 6 eigenvalues:", np.round(evals[:6], 8))

    N = 2**num_qubits
    x_vals, _, _ = jlp_grids(num_qubits, x_max)
    gaussian = np.exp(-0.5 * x_vals**2).astype(complex)
    gaussian /= np.linalg.norm(gaussian)

    v0 = evecs[:, 0]
    overlap = float(abs(np.vdot(v0, state))**2)
    print("Overlap(prepared state, discrete ground):", overlap)

# Time evolution

In [48]:
estimator = StatevectorEstimator()

In [49]:
# this is the list for x operator
x_op_list = [('I' * (num_qubits - i - 1) + 'Z' + 'I' * i, - 2 ** i * x_max / (2 ** num_qubits - 1)) for i in range(num_qubits)]
x_op = SparsePauliOp.from_list(x_op_list)
x_sq_op = x_op @ x_op

In [55]:
job = estimator.run([(qc, x_sq_op)])

In [56]:
job.result()[0].data.evs

array(0.35168074)